# 데이터 탐색 — 요약 통계량 · 상관관계 · plotly 시각화

수업 중 작성한 필기 3개(`Data_analysis02` · `통계분석연습` · `plotly`)를 **실행 가능하도록 하나로 정리한 노트**입니다.
(교재 `데이터분석` p.34~46 · p.33 비교/논리 연산자는 Day 017 에서 정리)

각 단원은 `개념 설명 → 예제 코드` 순서로 구성되어 있고,
원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

> 실습 데이터 `ad_performance.csv` · `weather.csv` · `tips.csv` 는 이 노트북과 **같은 폴더**에 있어야 합니다.
> 4단원은 `plotly` 가 필요합니다 → `pip install plotly`

## 0. 데이터 탐색이란

전처리로 데이터를 손봤다면, 이제 **데이터가 어떻게 생겼는지 파악**할 차례다.

| 단원 | 질문 | 도구 |
|------|------|------|
| 1~2. 통계로 요약하기 | 값들이 **어디에 모여 있고 얼마나 퍼져 있나?** | `describe()` · `mean()` · `std()` … |
| 3. 상관관계 분석 | 두 값이 **같이 움직이나?** | `corr()` |
| 4. 탐색적 데이터 분석(EDA) | **눈으로 보면** 어떤 모양인가? | `plotly.express` |

## 1. 통계로 요약하기 (교재 p.34~38)

### 1-1. 요약(기술) 통계량 — `df.describe()`

**데이터 분포의 특징을 나타내는 통계량.** 전체적인 특징과 분포를 **빠르게** 파악할 때 쓴다.
숫자 컬럼이면 기본으로 아래 8가지를 계산해 준다.

| 행 이름 | 뜻 |
|---------|-----|
| `count` | 데이터 개수 (**NaN 제외**) |
| `mean` | 평균값 |
| `std` | 표준편차 (데이터가 퍼져 있는 정도) |
| `min` | 최솟값 |
| `25%` | 1 사분위수 (하위 25% 지점의 값) |
| `50%` | 중앙값 (median, 2 사분위수) |
| `75%` | 3 사분위수 (하위 75% 지점의 값) |
| `max` | 최댓값 |

> `describe()` 의 결과도 **DataFrame** 이다. 그래서 `df.loc[행, 열]` 로 원하는 값 하나를 꺼낼 수 있다.

In [ ]:
import pandas as pd

path = "./ad_performance.csv"      # 노트북과 같은 폴더
df = pd.read_csv(path)

print("모양 :", df.shape)

df_state = df.describe()           # 요약 통계량 (이것도 DataFrame)
print(df_state.round(1).to_string())

### 1-2. 평균 (교재 p.35)

"광고 캠페인 하나에 **평균적으로 얼마의 비용**을 쓰고 있는가?" → 행 `mean`, 컬럼 `Cost`

값을 구하는 방법은 두 가지다.

| 방법 | 쓰는 법 |
|------|---------|
| ① 요약 통계량에서 꺼내기 | `df.describe().loc['mean', 'Cost']` |
| ② 컬럼에 바로 함수 쓰기 | `df['Cost'].mean()` |

> 하나만 필요하면 ②가 빠르다. 여러 통계량을 한꺼번에 볼 땐 ①로 표를 한 번 만들어 두고 꺼내 쓴다.

In [ ]:
# 방법 1 : 요약 통계량 -> 인덱싱
df_cost = df_state.loc['mean', 'Cost']
print("방법 1 (describe) :", df_cost.round(1))

# 방법 2 : mean() 메서드
avg_cost = df['Cost'].mean()
print("방법 2 (mean)     :", avg_cost.round(1))

### 1-3. 최솟값 · 최댓값 (교재 p.36)

"매출의 **최소**는 얼마인가?" → 행 `min`, 컬럼 `Revenue`
"매출의 **최대**는 얼마인가?" → 행 `max`, 컬럼 `Revenue`

In [ ]:
min_rev1 = df_state.loc['min', 'Revenue']     # 방법 1
min_rev2 = df['Revenue'].min()                # 방법 2
print("최솟값 :", min_rev1, "/", min_rev2)

max_rev1 = df_state.loc['max', 'Revenue']
max_rev2 = df['Revenue'].max()
print("최댓값 :", max_rev1, "/", max_rev2)

# describe() 쪽은 실수(1999273.0), min()/max() 쪽은 원래 자료형(정수)으로 나온다

### 1-4. 분위수 · 중앙값 (교재 p.37)

"Revenue 컬럼의 **중간값(중앙값)** 은 얼마인가?" → 행 `50%`, 컬럼 `Revenue`

중앙값을 구하는 방법은 세 가지다. 결과는 모두 같다.

| 방법 | 쓰는 법 |
|------|---------|
| ① 요약 통계량 | `df_state.loc['50%', 'Revenue']` |
| ② `median()` | `df['Revenue'].median()` |
| ③ `quantile()` | `df['Revenue'].quantile(0.5)` — 0.5 가 중앙값 |

> **평균과 중앙값을 같이 보면** 분포의 모양을 짐작할 수 있다.
> 평균이 중앙값보다 **작으면** 작은 값 쪽으로 꼬리가 길다는 뜻이다.

In [ ]:
median_rev1 = df_state.loc['50%', 'Revenue']
median_rev2 = df['Revenue'].median()
median_rev3 = df['Revenue'].quantile(0.5)

print("중앙값 :", median_rev1, "/", median_rev2, "/", median_rev3)

print("-" * 40)

mean_rev = df['Revenue'].mean()
print("평균   :", mean_rev.round(1))
print("평균 - 중앙값 :", (mean_rev - median_rev2).round(1), " <- 평균이 더 작다")

### 1-5. 표준편차 (교재 p.38)

**표준편차** — 데이터들이 평균값에서 **평균적으로 얼마나 떨어져 있는가?**
데이터의 **일관성(변동성)** 을 보여 주는 지표다.

| 표준편차가 | 뜻 |
|-----------|-----|
| **작다** | 값들이 평균 주변에 옹기종기 모여 있다 → 성과가 꾸준하고 **예측 가능** |
| **크다** | 값들이 평균에서 멀리 넓게 흩어져 있다 → 성과가 들쭉날쭉하고 **예측하기 힘듦** |

In [ ]:
std_rev1 = df_state.loc['std', 'Revenue']
std_rev2 = df['Revenue'].std()
print(f"표준편차 : {std_rev1.round(1)} / {std_rev2.round(1)}")

# 평균 대비 얼마나 퍼져 있나 (변동계수 = 표준편차 / 평균)
print(f"평균 대비 : {(std_rev2 / df['Revenue'].mean() * 100).round(1)} %")

print("-" * 40)

# ⚠️ '행의 개수'를 셀 때 헷갈리기 쉬운 부분
print("len(df_state)                 :", len(df_state), " <- describe 표의 행 수 (통계량 8가지)")
print("len(df)                       :", len(df), " <- 실제 데이터 행 수")
print("df_state.loc['count','Revenue']:", df_state.loc['count', 'Revenue'], " <- NaN 을 뺀 데이터 개수")

> **📝 고친 부분**
> - `row = len(df_state['Revenue'])` 에 **"행의 개수 확인 (추천!)"** 이라는 주석이 붙어 있었는데, 결과는 **8** 이다.
>   `df_state` 는 describe 표라서 통계량 8가지(count~max)의 개수가 나온 것. 실제 데이터 행 수는 `len(df)` = **5000**,
>   NaN 을 뺀 개수는 `df_state.loc['count', 'Revenue']` = **5000.0** 이다.
> - `df_state['Revenue'].count()` 도 같은 이유로 **8** 이 나온다 (통계량 8개를 세는 것)
> - `avg_cost` 를 계산만 하고 `#printt(...)` 로 막혀 있어 결과가 안 보였다 → `printt` 오타를 고쳐 살림
> - 변수명 오타 `medien_rev2` → `median_rev2`, 주석 오타 `요약 통계망` → **요약 통계량**
> - 마크다운 제목이 "Q1,Q2,Q3 · IQR · 최소값 구하기" 였는데 내용은 최솟값/최댓값만이라 교재 목차(p.36)에 맞춤
> - 절대 경로 `C:\Users\Jinho\work\...` → 상대 경로 `./ad_performance.csv`

## 2. 연습 — 날씨 데이터로 요약 통계량 구하기 (`통계분석연습`)

기상청 날씨 데이터 100,000행 (`temp` 기온 · `hum` 습도 · `atm` 기압 · `speed` 풍속)으로
1단원의 방법을 그대로 반복한다. 각 문제를 **방법 1(describe) · 방법 2(메서드)** 로 풀고 결과가 같은지 확인한다.

In [ ]:
path = "./weather.csv"
df_w = pd.read_csv(path)

print("모양 :", df_w.shape)

w_state = df_w.describe()
print(w_state.round(1).to_string())

In [ ]:
# 1) temp 컬럼 평균
print("temp 평균   :", w_state.loc['mean', 'temp'].round(1), "/", df_w['temp'].mean().round(1))

# 2) hum 컬럼 최솟값 · 최댓값
print("hum 최솟값  :", w_state.loc['min', 'hum'], "/", df_w['hum'].min())
print("hum 최댓값  :", w_state.loc['max', 'hum'], "/", df_w['hum'].max())

# 3) atm 컬럼 중앙값
print("atm 중앙값  :", w_state.loc['50%', 'atm'], "/", df_w['atm'].median(), "/", df_w['atm'].quantile(0.5))

# 4) speed 컬럼 표준편차
print("speed 표준편차 :", w_state.loc['std', 'speed'].round(2), "/", df_w['speed'].std().round(2))

> **📝 고친 부분**
> - 제목은 "**atm** 컬럼에 대한 중간값" 인데 코드는 `speed` 의 중앙값을 구하고 있었다 → 제목대로 `atm` 으로 고침
> - 주석으로 막혀 있던 세 번째 방법 `quantile(0.5)` 를 살려 세 방법을 나란히 비교
> - 오타 `임포틑` → **임포트**, `tmep` → **temp**
> - 절대 경로 → 상대 경로 `./weather.csv`
> - 문제 5개를 셀 5개로 나눠 각각 `print` 두 번 + 구분선을 찍던 것을 한 셀로 모아 **한눈에** 비교되게 함

## 3. 상관관계 분석 (교재 p.39~40)

### 3-1. 개념

**두 변수 사이에 연관성이 있는지, 있다면 얼마나 강한지** 측정하는 분석 방법.
예) 광고비 지출이 매출액 증가에 어느 정도 영향이 있는지 파악

**상관계수(피어슨 상관계수)** — 두 변수 사이의 **선형 관계 정도**를 **-1 ~ +1** 사이 숫자로 나타낸 것

| 상관계수 | 뜻 | 그래프 모양 |
|---------|-----|------------|
| **+1 에 가까움** | 양의 상관 — 한쪽이 오르면 다른 쪽도 **오른다** | 오른쪽 위로 올라가는 직선 |
| **0 에 가까움** | 상관 없음 — 선형 관계가 거의 없다 | 흩어진 점 |
| **-1 에 가까움** | 음의 상관 — 한쪽이 오르면 다른 쪽은 **내린다** | 오른쪽 아래로 내려가는 직선 |

절댓값으로 강도를 읽는 일반적인 기준

| \|r\| | 해석 |
|--------|------|
| 0.0 ~ 0.1 | 거의 없음 |
| 0.1 ~ 0.3 | 약함 |
| 0.3 ~ 0.7 | 뚜렷함 |
| 0.7 ~ 1.0 | 강함 |

> ⚠️ **상관관계 ≠ 인과관계.** 상관은 "같이 움직이는가"만 볼 뿐 "**왜** 그런지"는 말해 주지 않는다.
> 아이스크림 판매량과 익사 사고는 상관이 높지만, 아이스크림이 원인이 아니라 **더운 날씨**가 둘 다 늘린 것이다.

### 3-2. 상관 행렬 — `df.corr(numeric_only=True)` (교재 p.40)

모든 숫자 컬럼끼리의 상관계수를 **표(상관 행렬)** 로 만들어 준다.

- 대각선은 자기 자신과의 상관이라 항상 **1.00**
- 표가 **대각선 기준으로 대칭**이다 (`A-B` 와 `B-A` 는 같은 값)
- `numeric_only=True` : 문자열 컬럼(`Date`, `Channel` …)을 빼고 숫자 컬럼만 계산

In [ ]:
# 광고 데이터 상관 행렬
df_corr1 = df.corr(numeric_only=True)
print(df_corr1.round(2).to_string())

### 3-3. 두 변수만 골라 보기

| 방법 | 쓰는 법 |
|------|---------|
| ① 상관 행렬에서 꺼내기 | `df.corr(numeric_only=True).loc['Cost', 'Revenue']` |
| ② 두 컬럼만 골라 행렬 만들기 | `df[['Cost', 'Revenue']].corr()` |
| ③ Series 끼리 바로 | `df['Cost'].corr(df['Revenue'])` |

In [ ]:
r1 = df_corr1.loc['Cost', 'Revenue']                 # 방법 1
r2 = df[['Cost', 'Revenue']].corr()                  # 방법 2 (2x2 표)
r3 = df['Cost'].corr(df['Revenue'])                  # 방법 3

print("방법 1 :", round(r1, 4))
print("방법 2 :")
print(r2.round(4))
print("방법 3 :", round(r3, 4))

> 광고 데이터는 모든 상관계수가 **0 근처**(최대 0.10)다.
> "광고비를 많이 쓰면 매출이 오른다"는 **선형 관계가 이 데이터에서는 보이지 않는다**는 뜻이다.
> (실습용으로 무작위 생성된 데이터라 그렇다)

In [ ]:
# 날씨 데이터 상관 행렬 — 여기는 뚜렷한 관계가 보인다
w_corr = df_w.corr()
print(w_corr.round(2).to_string())

print("-" * 40)

# 온도(temp)와 습도(hum)의 상관관계
print("temp - hum :", round(df_w['temp'].corr(df_w['hum']), 2), " <- 뚜렷한 음의 상관 (기온이 높을수록 습도가 낮다)")
print("hum  - atm :", round(df_w['hum'].corr(df_w['atm']), 2), " <- 뚜렷한 음의 상관")

> **📝 고친 부분**
> - `df.corr(['temp', 'hum'])` → **ValueError**. `corr()` 의 첫 번째 자리는 계산 방법(`method`: `'pearson'` 등)이라
>   컬럼 목록을 넣으면 안 된다. 컬럼을 **먼저 고른 뒤** `df[['temp', 'hum']].corr()` 로 부르거나 `df['temp'].corr(df['hum'])`
> - `df.corr.loc['Cost', 'Revenue']` → **AttributeError: 'function' object has no attribute 'loc'**.
>   `corr` 는 **메서드**라 괄호로 **먼저 호출**해야 표가 생긴다 → `df.corr(numeric_only=True).loc['Cost', 'Revenue']`
> - 위 셀의 주석 버전 `df['Cost','Revenue']` 도 대괄호가 한 겹이라 KeyError. 여러 컬럼은 **두 겹** `df[['Cost','Revenue']]`
> - 필기의 "**오른쪽 하향하는 계수는 의미가 없다**" 는 틀린 설명이다. 오른쪽 아래로 내려가는 것은 **음의 상관**이고,
>   `temp`-`hum` 처럼 **-0.46 도 뚜렷한 관계**다. 의미가 없는 것은 **0 에 가까운** 경우다.
> - `상관 행열` → **상관 행렬**
> - 온도-습도 상관 분석 셀이 에러로 끝나 비어 있었다 → 결과와 해석까지 채움

## 4. 탐색적 데이터 분석 — EDA (교재 p.41~46)

**EDA(Exploratory Data Analysis)** — **시각화를 통해** 데이터의 특징과 패턴을 이해하는 것.

**시각화(visualization)** 는 데이터를 도표나 차트로 정리하는 것이다.

- 데이터의 특징을 **쉽게** 파악할 수 있다
- 분석 결과를 상대방에게 **효과적으로** 전달할 수 있다

### 4-1. 파이썬 시각화 라이브러리 (교재 p.42~45)

| 라이브러리 | 특징 | 불러오기 |
|-----------|------|---------|
| **matplotlib** | 파이썬의 대표적인 시각화 라이브러리 | `import matplotlib.pyplot as plt` |
| **seaborn** | matplotlib 기반, **통계 데이터** 시각화 | `import seaborn as sns` |
| **plotly** | **웹 브라우저 기반 인터랙티브** — 마우스를 올리면 값 표시(툴팁), 드래그로 확대·축소, 범례 클릭으로 필터링 | `import plotly.express as px` |

주로 쓰는 시각화 도구 — 직선 그래프 · 막대 그래프 · 원 그래프 · 히스토그램 · 박스 플롯 · 산점도 · 히트맵

### 4-2. 직선 그래프 — `px.line()` (교재 p.46)

**두 개 이상의 데이터 포인트를 직선으로 연결**해 연속적인 변화와 추세를 보여 준다.

```python
px.line(data_frame, x, y, color)
```

| 하이퍼파라미터 | 뜻 |
|---------------|-----|
| `data_frame` | 데이터프레임 |
| `x`, `y` | 시각화할 컬럼 |
| `color` | 새로운 변수를 추가해 **색상으로 구분** |

### 4-3. 실습 데이터 — `tips.csv`

식당 손님 244명의 계산 금액과 팁을 기록한 데이터.

| 컬럼 | 뜻 |
|------|-----|
| `total_bill` | 총 계산 금액 (달러) |
| `tip` | 팁 (달러) |
| `sex` | 계산한 사람의 성별 |
| `smoker` | 흡연석 여부 |
| `day` | 요일 (Thur · Fri · Sat · Sun) |
| `time` | 점심(Lunch) / 저녁(Dinner) |
| `size` | 일행 수 |

In [ ]:
import plotly.express as px

path = "./tips.csv"
df_tips = pd.read_csv(path)

print("모양 :", df_tips.shape)
print(df_tips.head())

print("-" * 40)

# 점심 / 저녁 손님 수
print(df_tips['time'].value_counts())

> **직선 그래프는 먼저 정렬해야 한다.**
> `px.line()` 은 **행 순서대로** 점을 잇는다. `total_bill` 이 뒤죽박죽인 채로 그리면 선이 왔다 갔다 하며 엉킨다.
> 그래서 x 축 컬럼으로 **오름차순 정렬**한 뒤 그린다.

In [ ]:
# x 축 기준 오름차순 정렬 (정렬하지 않으면 선이 엉킨다)
df_sorted = df_tips.sort_values(by="total_bill")

fig = px.line(df_sorted, x="total_bill", y="tip",
              title="계산 금액이 클수록 팁도 커질까?")
fig.show()

print("상관계수 total_bill - tip :", round(df_tips['total_bill'].corr(df_tips['tip']), 2))

In [ ]:
# color : 새로운 변수를 추가해 색상으로 구분 (교재 p.46 하이퍼파라미터)
fig2 = px.line(df_sorted, x="total_bill", y="tip", color="time",
               title="점심 / 저녁으로 나눠 보기")
fig2.show()

# 그래프를 HTML 파일로 저장하면 브라우저에서 그대로 열린다 (필요할 때 주석 해제)
# fig2.write_html("tips_line.html")

> **📝 고친 부분**
> - 절대 경로 `C:\Users\Jinho\work\tips.csv` → 상대 경로 `./tips.csv`
> - `warnings.filterwarnings("ignore")` 로 **모든 경고를 끄던 것**을 뺐다.
>   경고는 "지금은 되지만 곧 안 될 수 있다"는 신호라, 전부 끄면 문제가 생겨도 원인을 놓친다. 필요할 때 특정 경고만 끈다.
> - `cont_time = df_sorted['time'].value_counts()` 를 그래프와 같은 셀에서 찍어 출력이 섞였다 → 데이터 확인 셀로 분리
> - 교재 p.46 의 **`color` 하이퍼파라미터** 예제가 필기에 없어 추가 (`time` 으로 점심/저녁 구분)
> - 그래프에 `title` 을 달아 무엇을 보려는 그래프인지 드러나게 함
> - `display(df)` → 244행을 다 찍지 않고 `head()`

## 5. 오늘 정리

- `df.describe()` 는 **8가지 요약 통계량**(count·mean·std·min·25%·50%·75%·max)을 DataFrame 으로 돌려준다.
  값 하나는 `df.describe().loc['행', '열']` 또는 `df['열'].mean()` 같은 메서드로 꺼낸다.
- `len(df.describe())` 는 **8** 이다 — 데이터 행 수는 `len(df)`.
- 중앙값은 `loc['50%']` · `median()` · `quantile(0.5)` 세 가지 방법이 모두 같다.
- 표준편차가 **작으면 꾸준하고 예측 가능**, **크면 들쭉날쭉**.
- 상관계수는 **-1 ~ +1**. 0 에 가까우면 관계가 없고, **음수도 뚜렷한 관계**다. 상관 ≠ 인과.
- `df.corr(numeric_only=True)` → 상관 행렬. `corr` 는 메서드라 **괄호부터** 붙인다. 두 컬럼만은 `df['A'].corr(df['B'])`.
- 직선 그래프는 x 축으로 **먼저 정렬**하고 그린다. `color=` 로 그룹을 색으로 나눈다.